# BB-code bit-flip-noise Monte Carlo for symmetry matching + K-MWM

This notebook follows the style of the other notebooks in `examples/tutorials/ldpc`.

Goal:
- use the existing BB / CSS utilities in `LatticeAlgorithms.jl`;
- build the symmetry matching subproblems for code-capacity bit-flip noise;
- apply a user-supplied K-MWM decoder to each symmetry subproblem;
- extract logical information using the cylinder trick;
- plot logical failure rate versus physical bit-flip probability.

Important:
- I avoid local copies of existing repo utilities such as `gf2_nullspace`, `css_logicals`, `bb_check_matrices`, `bp_osd_cs_decode`, etc.
- The only helper functions below are new symmetry-matching glue code. Once stable, these should be moved into `src/bb_symatch.jl` and exported.

In [ ]:
using Distributed
using JLD2
using Plots

num_cores = length(Sys.cpu_info())
if nprocs() == 1
    addprocs(num_cores; exeflags=`--project=$(Base.active_project())`)
end ;

@everywhere begin
    using LatticeAlgorithms
    using LinearAlgebra
    using SparseArrays
    using Random
    using Dates
end

In [ ]:
println("num_cores = $(num_cores)")

## Code family definitions

The Gross-code BB polynomial convention follows the repo constructor

```julia
HX, HZ = bb_check_matrices(l, m, A_terms, B_terms)
```

where
\[
H_X = [A\ B],\qquad H_Z = [B^T\ A^T].
\]

Please verify the `two_gross` and `cyclic_hgp` entries against the exact specs you want to study.  I left them in the same BB-constructor format so that no separate local code constructor is introduced.

In [ ]:
@everywhere begin
    const BB_CODE_SPECS = Dict(
        # Gross [[144,12,12]]
        # Common convention: l=6, m=12,
        # A = 1 + x^3 + y, B = 1 + y^3 + x.
        :gross => (
            l = 6,
            m = 12,
            A_terms = [(:I, 0), (:x, 3), (:y, 1)],
            B_terms = [(:I, 0), (:y, 3), (:x, 1)],
            label = "[[144,12,12]] Gross",
        ),

        # Placeholder in BB format.
        # Replace by the exact two-Gross polynomial if your repo has a named constructor.
        :two_gross => (
            l = 12,
            m = 12,
            A_terms = [(:I, 0), (:x, 3), (:y, 1)],
            B_terms = [(:I, 0), (:y, 3), (:x, 1)],
            label = "two-Gross placeholder",
        ),

        # Placeholder in BB/cyclic-HGP-compatible format.
        # Replace by the exact cyclic-HGP constructor/spec used in your repo.
        :cyclic_hgp => (
            l = 12,
            m = 12,
            A_terms = [(:I, 0), (:x, 1), (:y, 2)],
            B_terms = [(:I, 0), (:x, 2), (:y, 1)],
            label = "cyclic-HGP placeholder",
        ),
    )

    function bb_code_from_spec(spec)
        HX, HZ = bb_check_matrices(spec.l, spec.m, spec.A_terms, spec.B_terms)
        X_basis, Z_basis = css_logicals(HX, HZ)
        return (
            HX = HX,
            HZ = HZ,
            X_basis = X_basis,
            Z_basis = Z_basis,
            n = size(HX, 2),
            num_checks = size(HZ, 1),
            k = size(X_basis, 1),
        )
    end
end

## Symmetry-matching glue code

For bit-flip noise, the measured syndrome is

\[
s = H_Z e_X \pmod 2.
\]

A check symmetry is a binary vector \(u\) with

\[
u^T H_Z = 0.
\]

For each such \(u\), the selected checks define a toric-code-like matching subproblem.  This notebook constructs the corresponding restricted syndrome equation

\[
H_Z[u,:]\, e = s[u],
\]

and passes it to the K-MWM decoder hook.

In [ ]:
@everywhere begin
    function check_symmetry_basis(H::AbstractMatrix{<:Integer})
        # Rows are independent check symmetries u satisfying u^T H = 0.
        return gf2_nullspace(transpose(H))
    end

    function selected_rows(u::AbstractVector{<:Integer})
        return Int64.(findall(!iszero, vec(mod.(u, 2))))
    end

    function bb_check_coord(row_idx::Int, l::Int, m::Int)
        # BB check rows are indexed over an l × m translation grid.
        # This matches the row indexing induced by kron(S_l, I_m), kron(I_l, S_m).
        a = fld(row_idx - 1, m) + 1
        b = mod(row_idx - 1, m) + 1
        return a, b
    end

    function cylinder_rows(
        rows::Vector{Int64},
        l::Int,
        m::Int;
        direction::Symbol = :x,
        offset::Int = 0,
        width::Union{Nothing, Int} = nothing,
    )
        # Select one cylinder from the selected symmetry checks.
        # direction=:x cuts along the l direction; direction=:y cuts along the m direction.
        L = direction == :x ? l : m
        W = isnothing(width) ? fld(L, 2) : width
        chosen = Int64[]
        for r in rows
            a, b = bb_check_coord(r, l, m)
            coord = direction == :x ? a : b
            t = mod(coord - 1 - offset, L) + 1
            if 1 <= t <= W
                push!(chosen, r)
            end
        end
        return chosen
    end

    function cylinder_logical(H::AbstractMatrix{<:Integer}, cyl_rows::Vector{Int64})
        # Product of checks in one cylinder gives the boundary/logical ribbon.
        if isempty(cyl_rows)
            return zeros(Int64, size(H, 2))
        end
        return vec(mod.(sum(H[cyl_rows, :]; dims=1), 2))
    end

    function syndrome_from_bitflip(HZ::AbstractMatrix{<:Integer}, e::AbstractVector{<:Integer})
        return vec(mod.(HZ * mod.(Int64.(e), 2), 2))
    end

    function random_bitflip(n::Int, p::Real)
        return Int64.(rand(n) .< p)
    end

    function is_logical_failure(
        HX::AbstractMatrix{<:Integer},
        HZ::AbstractMatrix{<:Integer},
        residual_x::AbstractVector{<:Integer},
        Z_basis::AbstractMatrix{<:Integer},
    )
        # A residual X error is harmless iff it commutes with every Z logical.
        # Equivalently, its symplectic pairing with the Z logical basis is zero.
        if size(Z_basis, 1) == 0
            return false
        end
        logical_syndrome = mod.(Z_basis * mod.(Int64.(residual_x), 2), 2)
        return any(!iszero, logical_syndrome)
    end
end

## K-MWM decoder hook

Replace the body of `k_mwm_binary_decode` by the actual K-MWM routine in your repo.

Expected return format:

```julia
candidates = [
    (error = e1, weight = w1),
    (error = e2, weight = w2),
    ...
]
```

where each `error` is a length-`n` binary correction satisfying `H * error == s mod 2`.

The temporary fallback below uses `bp_osd_cs_decode` and returns one candidate only.  It is only there so the rest of the notebook can be tested before wiring in K-MWM.

In [ ]:
@everywhere begin
    function k_mwm_binary_decode(
        H::AbstractMatrix{<:Integer},
        s::AbstractVector{<:Integer},
        p::Real;
        K::Int = 1,
    )
        # TODO: replace this fallback with your exact K-MWM decoder.
        #
        # Example intended shape:
        #   out = k_mwm_decode(H, s, p; K=K)
        #   return [(error = out.errors[i], weight = out.weights[i]) for i in 1:length(out.errors)]
        #
        # Fallback: BP+OSD-0 single candidate, using existing repo utility.
        res = bp_osd_cs_decode(H, s, p; osd_order=0, max_iter=size(H, 2))
        return [(error = res.error, weight = count(!iszero, res.error))]
    end
end

## Symmetry matching with cylinder trick

For each independent symmetry \(u_a\):

1. restrict to the selected check rows;
2. run K-MWM on that restricted syndrome;
3. form a cylinder logical ribbon from the product of checks in one cylinder;
4. record the parity \(e_i \cdot L_a\) for each candidate.

The simple fusion rule below picks, for each symmetry, the lowest-weight candidate and XORs the corresponding cylinder-ribbon correction when its parity is odd.  This is intentionally minimal; it gives you a clean place to add simplex-symatch / overcomplete-symmetry fusion later.

In [ ]:
@everywhere begin
    function symmetry_graphs_for_bb(
        HZ::AbstractMatrix{<:Integer},
        l::Int,
        m::Int;
        max_num_symmetries::Union{Nothing, Int} = nothing,
        cylinder_direction::Symbol = :x,
    )
        U = check_symmetry_basis(HZ)
        nsym = size(U, 1)
        if !isnothing(max_num_symmetries)
            nsym = min(nsym, max_num_symmetries)
        end

        graphs = []
        for a in 1:nsym
            rows = selected_rows(view(U, a, :))
            cyl = cylinder_rows(rows, l, m; direction=cylinder_direction)
            L = cylinder_logical(HZ, cyl)
            push!(graphs, (
                id = a,
                symmetry = vec(U[a, :]),
                rows = rows,
                H = HZ[rows, :],
                cylinder_rows = cyl,
                cylinder_logical = L,
            ))
        end
        return graphs
    end

    function symatch_kmwm_decode(
        HZ::AbstractMatrix{<:Integer},
        s::AbstractVector{<:Integer},
        graphs,
        p::Real;
        K::Int = 1,
    )
        n = size(HZ, 2)
        correction = zeros(Int64, n)
        parity_records = Vector{Any}()

        for g in graphs
            sg = s[g.rows]
            candidates = k_mwm_binary_decode(g.H, sg, p; K=K)

            # First candidate = current hard decision for this symmetry.
            best = candidates[1]
            parity = mod(sum(best.error .* g.cylinder_logical), 2)

            if parity == 1
                correction = mod.(correction .+ g.cylinder_logical, 2)
            end

            push!(parity_records, (
                graph_id = g.id,
                parity = parity,
                candidates = candidates,
                candidate_parities = [mod(sum(c.error .* g.cylinder_logical), 2) for c in candidates],
            ))
        end

        return (correction = correction, parity_records = parity_records)
    end
end

## Monte Carlo driver

This reproduces the usual Fig.-1 style plot: logical failure rate versus physical bit-flip probability.

For the paper-like comparison you probably want:
- `K = 1` baseline symatch;
- larger `K` values for your K-MWM variant;
- later, BP-reweighted symmetry edge weights / simplex fusion.

In [ ]:
@everywhere begin
    function run_symatch_trials(
        code_key::Symbol,
        p::Real,
        num_samples_each_core::Int;
        K::Int = 1,
        max_num_symmetries::Union{Nothing, Int} = nothing,
        cylinder_direction::Symbol = :x,
        rng_seed::Int = 0,
    )
        if rng_seed != 0
            Random.seed!(rng_seed + myid())
        end

        spec = BB_CODE_SPECS[code_key]
        code = bb_code_from_spec(spec)
        graphs = symmetry_graphs_for_bb(
            code.HZ,
            spec.l,
            spec.m;
            max_num_symmetries=max_num_symmetries,
            cylinder_direction=cylinder_direction,
        )

        num_fail = 0
        total_decode_time = 0.0

        for _ in 1:num_samples_each_core
            e = random_bitflip(code.n, p)
            s = syndrome_from_bitflip(code.HZ, e)

            total_decode_time += @elapsed begin
                dec = symatch_kmwm_decode(code.HZ, s, graphs, p; K=K)
            end

            residual = mod.(e .+ dec.correction, 2)
            num_fail += is_logical_failure(code.HX, code.HZ, residual, code.Z_basis) ? 1 : 0
        end

        return (num_fail = num_fail, total_decode_time = total_decode_time)
    end
end

In [ ]:
# Parameters for a quick test. Increase num_samples for real data.
code_keys = [:gross]  # later: [:gross, :two_gross, :cyclic_hgp]
Krange = [1, 2, 5, 10]
prange = collect(0.01:0.01:0.08)

num_super_samples = 1
num_samples = 1_000

num_samples_each_core = Int(ceil(num_samples / num_cores))
num_samples = num_samples_each_core * num_cores
num_total_samples = num_super_samples * num_samples

println((num_samples_each_core = num_samples_each_core, num_total_samples = num_total_samples))

In [ ]:
results = Dict()

total_t = @elapsed for code_key in code_keys
    spec = BB_CODE_SPECS[code_key]
    code = bb_code_from_spec(spec)
    println("code = $(spec.label), n = $(code.n), k = $(code.k)")

    # For Gross [[144,12,12]], start with k/2 = 6 independent symmetry graphs.
    max_num_symmetries = div(code.k, 2)

    for K in Krange
        for p in prange
            failures = 0
            decode_time = 0.0

            for ind in 1:num_super_samples
                worker_results = pmap(1:num_cores) do _
                    run_symatch_trials(
                        code_key,
                        p,
                        num_samples_each_core;
                        K = K,
                        max_num_symmetries = max_num_symmetries,
                        cylinder_direction = :x,
                        rng_seed = 1234 + 1000 * ind,
                    )
                end

                failures += sum(x.num_fail for x in worker_results)
                decode_time += sum(x.total_decode_time for x in worker_results)

                println("$(now()) code=$code_key K=$K p=$p super=$ind/$num_super_samples failures=$failures")
            end

            results[(code_key, K, p)] = (
                logical_error_rate = failures / num_total_samples,
                mean_decode_time = decode_time / num_total_samples,
                failures = failures,
                num_samples = num_total_samples,
            )
        end
    end
end

println("total wall time = $total_t")

In [ ]:
fn = "bb_symatch_kmwm_bitflip_$(minimum(prange))_$(maximum(prange))_$(maximum(Krange))_$(num_total_samples).jld2"
jldsave(fn;
    code_keys = code_keys,
    Krange = Krange,
    prange = prange,
    num_samples = num_total_samples,
    results = results,
)
println(fn)

## Plot

In [ ]:
linecolors = get_color_palette(:auto, plot_color(:white))

plots = []
for code_key in code_keys
    g = plot()
    for (iK, K) in enumerate(Krange)
        ys = [results[(code_key, K, p)].logical_error_rate for p in prange]
        ns = [results[(code_key, K, p)].num_samples for p in prange]
        yerr = sqrt.(ys .* (1 .- ys) ./ ns)
        plot!(
            g,
            prange,
            ys;
            marker = :circle,
            yerr = yerr,
            label = "K=$K",
            color = linecolors[iK],
        )
    end
    plot!(
        g;
        xlabel = "physical bit-flip probability p",
        ylabel = "logical failure rate",
        title = string(code_key),
        yscale = :log10,
        legend = :bottomright,
        size = (650, 450),
    )
    push!(plots, g)
end

plot(plots..., layout = (length(plots), 1))

## Next implementation notes

The current notebook is deliberately minimal and repo-style.

Recommended next move:
1. move the symmetry functions into `src/bb_symatch.jl`;
2. export `symmetry_graphs_for_bb` and `symatch_kmwm_decode`;
3. replace `k_mwm_binary_decode` with the actual K-MWM routine;
4. add overcomplete translated symmetries and simplex fusion;
5. add BP-reweighted priors for the symmetry subproblems.